# Text, image and optional audio input

## Goal

Send text and an uploaded image; optionally record and submit a short audio sample.

## Setup

Select the **Python (LangChain OpenAI)** kernel and run cells from top to bottom.
`OPENAI_API_KEY` and `OPENAI_MODEL` are read only from this course's root `.env`.
Shared settings are in `lesson_support.py`: up to 1024 output tokens, a 30-second
HTTP timeout, no HTTP retries, and LangSmith tracing disabled.
`make_model("another-api-model-id")` creates a different model with the same key.
The default text settings are intended for `gpt-5.6-luna`.
Creating a model does not call the API; `invoke` and consuming `stream` make paid requests.


## Text input

https://platform.openai.com/docs/models

In [ ]:
from pathlib import Path
import sys

# Find this course's root, whether Jupyter starts at the root or in notebooks/.
current_dir = Path.cwd().resolve()
course_root = next(
    (folder for folder in (current_dir, *current_dir.parents)
     if (folder / "course.json").is_file() and (folder / "lesson_support.py").is_file()),
    None,
)
if course_root is None:
    raise RuntimeError("Start Jupyter from the course root, as described in README.md.")
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))
from lesson_support import make_model

model = make_model()
print("Model:", model.model_name)


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

Upload a sample PNG/JPEG, then run the following cells. Without an uploaded file, the image request is skipped.

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept=".png,.jpg,.jpeg", multiple=False)
display(uploader)


In [ ]:
# Show file metadata only, without dumping binary content.
for uploaded in uploader.value:
    print(uploaded["name"], uploaded["size"], "bytes")


In [ ]:
import base64

img_b64 = None
image_mime = None
if uploader.value:
    uploaded_file = uploader.value[0]
    image_mime = uploaded_file["type"]
    if image_mime not in {"image/png", "image/jpeg"}:
        raise ValueError("Upload a PNG or JPEG image")
    img_b64 = base64.b64encode(bytes(uploaded_file["content"])).decode("ascii")
else:
    print("No image uploaded: skipping the image request.")


In [ ]:
if img_b64 is not None:
    multimodal_question = HumanMessage(content=[
        {"type": "text", "text": "Tell me about this capital"},
        {"type": "image", "base64": img_b64, "mime_type": image_mime},
    ])
    response = agent.invoke({"messages": [multimodal_question]})
    print(response["messages"][-1].content)


## Audio input (optional)

The default text model does not accept audio. This section uses OPENAI_AUDIO_MODEL
from the root .env, or gpt-audio. Set audio_enabled = True explicitly to record five
seconds from your microphone; the next cell sends that recording to OpenAI.
By default, both recording and the audio request are skipped. Microphone access and
PortAudio support are platform-specific; see README.md before enabling recording.

In [ ]:
audio_enabled = False  # Enable explicitly to record from the microphone.
aud_b64 = None

if audio_enabled:
    import sounddevice as sd
    from scipy.io.wavfile import write
    import base64
    import io

    duration = 5
    sample_rate = 24000
    print("Recording for 5 seconds...")
    audio = sd.rec(duration * sample_rate, samplerate=sample_rate, channels=1, dtype="int16")
    sd.wait()
    buf = io.BytesIO()
    write(buf, sample_rate, audio)
    aud_b64 = base64.b64encode(buf.getvalue()).decode("ascii")
else:
    print("Microphone recording is disabled: skipping the audio request.")


In [ ]:
if aud_b64 is not None:
    # Use a separate audio model without reasoning_effort.
    audio_model = make_model(audio=True)
    audio_agent = create_agent(model=audio_model)
    multimodal_question = HumanMessage(content=[
        {"type": "text", "text": "Tell me about this audio file"},
        {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"},
    ])
    audio_response = audio_agent.invoke({"messages": [multimodal_question]})
    print(audio_response["messages"][-1].content)
